In [5]:
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

DATA = Path("../data")

df = pd.read_csv(DATA / 'ml_sessions.csv')

In [3]:
numeric = [
    'station_age_years', 'power_kw', 'num_bays', 'hour',
    'day_of_week', 'is_weekend', 'ambient_temp_c', 'grid_load_index',
    'start_soc_pct', 'battery_capacity_kwh'
]

categorical = ['connector_type', 'zone', 'vehicle_segment']

X, y = df[numeric + categorical], df['failed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.25, random_state = 42, stratify = y
)
print(X_train.shape)

(33750, 13)


In [7]:
prep = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown = 'ignore'), categorical)
])

prep.fit(X_train)
model = LogisticRegression(max_iter =1000)
model.fit(prep.transform(X_train), y_train)
hand_probs = model.predict_proba(prep.transform(X_test))[:, 1]
print("ROC_AUC by hand: ", round(roc_auc_score(y_test, hand_probs), 4))

ROC_AUC by hand:  0.6923


In [9]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('prep', ColumnTransformer([
        ('num', StandardScaler(), numeric),
        ('cat', OneHotEncoder(handle_unknown = 'ignore'), categorical),
    ])),
    ("model", LogisticRegression(max_iter = 1000)),
])

pipe.fit(X_train, y_train)
pipe_probs = pipe.predict_proba(X_test)[:, 1]
print("ROC-AUC with a pipeline: ", round(roc_auc_score(y_test, pipe_probs), 4))

ROC-AUC with a pipeline:  0.6923


## Let's look inside the pipeline


In [10]:
print(pipe.named_steps.keys())

dict_keys(['prep', 'model'])


In [12]:
print("coefficients learned: ", pipe.named_steps['model'].coef_)

coefficients learned:  [[ 0.25506654  0.11879148  0.06153968 -0.02092131  0.03184734 -0.03943069
   0.09389887  0.47958832  0.04032678 -0.04707053 -0.47783625  0.0929602
  -0.43201002  0.09463643 -0.09483182 -0.62679865 -0.22947333 -0.28818749
  -0.31903606 -0.31192044 -0.29526278 -0.4650972  -0.43628476 -0.44809826
  -0.09439989]]


In [ ]:
print("features the model saw: ", len(pipe[:-1]))

features the model saw:  25


In [20]:
len(pipe[:-1].get_feature_names_out())

25

In [22]:
pipe[:-1].get_feature_names_out()[-3:]

array(['cat__vehicle_segment_3W', 'cat__vehicle_segment_4W',
       'cat__vehicle_segment_Bus'], dtype=object)

In [23]:
from sklearn.pipeline import make_pipeline

quick = make_pipeline(StandardScaler(), LogisticRegression())


In [24]:
quick.steps

[('standardscaler', StandardScaler()),
 ('logisticregression', LogisticRegression())]

In [26]:
for name, function in quick.steps:
    print(name)

standardscaler
logisticregression


In [33]:
import numpy as np
split_aucs = []
for seed in range(8):
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size =0.25, random_state = seed,
        stratify = y_train
    )
    pipe.fit(X_tr, y_tr)
    probs = pipe.predict_proba(X_val)[:, 1]
    split_aucs.append(roc_auc_score(y_val, probs))

print(np.round(split_aucs, 4))


[0.6992 0.6822 0.676  0.6856 0.6875 0.7053 0.6867 0.6953]


In [34]:
min(split_aucs)

0.6759730751598831

In [35]:
max(split_aucs)

0.7053382172052088

## Cross Validation